# Kernel Functions and Implicit Mapping to Higher-Dimensional Feature Spaces

## Overview

A **kernel function** allows a learning algorithm (most notably a Support Vector Machine) to operate in a **high-dimensional feature space** without ever explicitly computing the coordinates of data points in that space. This is the celebrated **"kernel trick"**.

This notebook provides:
1. A **written explanation** of how a kernel function implicitly maps data into a higher-dimensional feature space $\mathcal{H}$ where linear separation becomes possible.
2. A **diagram** illustrating the mapping $\varphi : \mathcal{X} \to \mathcal{H}$ and why it is needed.
3. A **demonstration** confirming the kernel identity $K(x, x') = \langle \varphi(x),\, \varphi(x') \rangle_{\mathcal{H}}$.

---

## 1. Motivation – Why Do We Need a Feature Map?

Many real-world datasets are **not linearly separable** in the original input space $\mathcal{X} \subseteq \mathbb{R}^d$.
For example, two classes might be arranged in concentric rings — no straight line (hyperplane) can separate them.

The key insight is:

> *If we map the data to a sufficiently rich feature space $\mathcal{H}$, the transformed data may become **linearly separable** there, even though it was not in the original space.*

Formally, we define a mapping
$$\varphi : \mathcal{X} \longrightarrow \mathcal{H}$$
where $\mathcal{H}$ is a **Hilbert space** (a vector space equipped with an inner product $\langle \cdot, \cdot \rangle_{\mathcal{H}}$) that may have very high or even infinite dimension.

A linear classifier trained on $\{\varphi(x_i)\}$ in $\mathcal{H}$ corresponds to a **non-linear** classifier in the original space $\mathcal{X}$.

---

## 2. The Kernel Trick

Computing $\varphi(x)$ explicitly can be prohibitively expensive when $\mathcal{H}$ is high-dimensional.
Many learning algorithms (SVMs, kernel PCA, Gaussian processes, …) only ever need **inner products** between pairs of mapped data points:
$$\langle \varphi(x),\, \varphi(x') \rangle_{\mathcal{H}}$$

A **kernel function** computes this inner product *directly* from the original inputs, without materialising $\varphi(x)$:

$$\boxed{K(x, x') = \langle \varphi(x),\, \varphi(x') \rangle_{\mathcal{H}}}$$

This is the **kernel trick**: replace every occurrence of $\langle \varphi(x_i), \varphi(x_j)\rangle$ in the algorithm with $K(x_i, x_j)$, evaluated cheaply in the original space.

### Mercer's Theorem
Any symmetric, positive semi-definite function $K : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ is a valid kernel, meaning a corresponding feature map $\varphi$ and Hilbert space $\mathcal{H}$ always exist (the **Reproducing Kernel Hilbert Space**, RKHS), even if we never write them down explicitly.

### Common Kernels
| Kernel | Formula | Notes |
|--------|---------|-------|
| Linear | $K(x, x') = x^\top x'$ | $\varphi$ is the identity |
| Polynomial | $K(x, x') = (x^\top x' + c)^d$ | maps to degree-$d$ monomials |
| RBF / Gaussian | $K(x, x') = \exp\!\left(-\dfrac{\|x - x'\|^2}{2\sigma^2}\right)$ | infinite-dimensional $\mathcal{H}$ |
| Sigmoid | $K(x, x') = \tanh(\alpha\, x^\top x' + c)$ | neural-network analogy |

---

## 3. Worked Example – Polynomial Kernel

Let $\mathcal{X} = \mathbb{R}^2$ and take the degree-2 polynomial kernel:
$$K(x, x') = (x^\top x')^2$$

For $x = (x_1, x_2)^\top$ and $x' = (x_1', x_2')^\top$:
$$K(x, x') = (x_1 x_1' + x_2 x_2')^2
= x_1^2 (x_1')^2 + 2 x_1 x_2 x_1' x_2' + x_2^2 (x_2')^2$$

This equals $\langle \varphi(x), \varphi(x') \rangle_{\mathbb{R}^3}$ with the **explicit feature map**
$$\varphi(x) = \bigl(x_1^2,\; \sqrt{2}\, x_1 x_2,\; x_2^2\bigr)^\top$$

We lifted 2D input to 3D feature space. For degree $d$ in $\mathbb{R}^n$ the feature dimension grows as $\binom{n+d}{d}$ — the kernel avoids this explosion.

---

## 4. Diagram – Input Space $\mathcal{X}$ → Feature Space $\mathcal{H}$

The cell below produces the conceptual diagram described in this section.

**Left panel** – Input space $\mathcal{X}$ ($\mathbb{R}^2$): data is arranged in two concentric rings and is **not** linearly separable.

**Middle arrow** – The (implicit) feature map $\varphi : \mathcal{X} \to \mathcal{H}$.

**Right panel** – Feature space $\mathcal{H}$ ($\mathbb{R}^3$ projection shown): after mapping, the two classes become linearly separable by a hyperplane.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── reproducibility ────────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

# ── generate concentric-ring data (not linearly separable in 2D) ───────────────
n = 200

# inner ring  (class +1)
theta1 = rng.uniform(0, 2 * np.pi, n)
r1     = rng.normal(1.0, 0.15, n)
X1     = np.c_[r1 * np.cos(theta1), r1 * np.sin(theta1)]

# outer ring  (class -1)
theta2 = rng.uniform(0, 2 * np.pi, n)
r2     = rng.normal(2.5, 0.15, n)
X2     = np.c_[r2 * np.cos(theta2), r2 * np.sin(theta2)]

X = np.vstack([X1, X2])
y = np.hstack([np.ones(n), -np.ones(n)])

# ── explicit feature map for the degree-2 kernel (1,x1,x2,x1²,x1x2,x2²) ──────
# We use only the three quadratic features for a clean 3-D visualisation
def phi(x):
    """Explicit feature map: R^2 -> R^3  (x1^2, sqrt(2)*x1*x2, x2^2)"""
    return np.c_[x[:, 0]**2, np.sqrt(2) * x[:, 0] * x[:, 1], x[:, 1]**2]

Phi  = phi(X)   # shape (2n, 3)
Phi1 = Phi[y ==  1]
Phi2 = Phi[y == -1]

# ── separating plane in feature space ─────────────────────────────────────────
# The radial boundary r = r_mid  translates to  x1^2 + x2^2 = r_mid^2
# In feature space (phi1, phi2, phi3) this is  phi1 + phi3 = r_mid^2  (a plane)
r_mid = 1.75
xx, yy = np.meshgrid(np.linspace(-0.2, 7, 30), np.linspace(-5, 5, 30))
zz = r_mid**2 - xx   # phi1 + phi3 = r_mid^2  =>  phi3 = r_mid^2 - phi1

# ── plotting ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 6))
fig.patch.set_facecolor('#f8f9fa')

gs = GridSpec(1, 5, figure=fig, width_ratios=[3, 0.5, 0.6, 0.5, 3])

# ── LEFT: Input space X ───────────────────────────────────────────────────────
ax_in = fig.add_subplot(gs[0, 0])
ax_in.set_facecolor('#ffffff')
ax_in.scatter(X1[:, 0], X1[:, 1], c='#e07b54', s=12, alpha=0.7, label='Class +1 (inner ring)')
ax_in.scatter(X2[:, 0], X2[:, 1], c='#4a90d9', s=12, alpha=0.7, label='Class -1 (outer ring)')
ax_in.set_title('Input Space $\\mathcal{X}$ ($\\mathbb{R}^2$)',
                fontsize=13, fontweight='bold', pad=10)
ax_in.set_xlabel('$x_1$', fontsize=12)
ax_in.set_ylabel('$x_2$', fontsize=12)
ax_in.legend(fontsize=9, loc='upper right')
ax_in.set_aspect('equal')
ax_in.spines[['top', 'right']].set_visible(False)
ax_in.text(0, -3.6, 'Not linearly separable', ha='center', fontsize=10,
           color='#c0392b', style='italic', fontweight='bold')

# add a dashed circle hint
circle = plt.Circle((0, 0), r_mid, color='gray', fill=False, linestyle='--',
                     linewidth=1.5, alpha=0.6)
ax_in.add_patch(circle)

# ── MIDDLE: arrow + annotation ────────────────────────────────────────────────
ax_arr = fig.add_subplot(gs[0, 1:4])
ax_arr.axis('off')

# Arrow
ax_arr.annotate(
    '',
    xy=(0.85, 0.5), xycoords='axes fraction',
    xytext=(0.15, 0.5), textcoords='axes fraction',
    arrowprops=dict(arrowstyle='->', color='#2c3e50',
                    lw=3, mutation_scale=25)
)

# Feature map label above arrow
ax_arr.text(0.5, 0.65, r'$\varphi : \mathcal{X} \to \mathcal{H}$',
            ha='center', va='bottom', fontsize=14,
            transform=ax_arr.transAxes, color='#2c3e50', fontweight='bold')

# Kernel trick label below arrow
ax_arr.text(0.5, 0.35,
            r'$K(x,x^{\prime})=\langle\varphi(x),\varphi(x^{\prime})\rangle_{\mathcal{H}}$',
            ha='center', va='top', fontsize=11,
            transform=ax_arr.transAxes, color='#7f8c8d',
            bbox=dict(boxstyle='round,pad=0.4', fc='#ecf0f1', ec='#bdc3c7', lw=1))

ax_arr.text(0.5, 0.18, '(kernel trick: no\nexplicit $\\varphi$ needed)',
            ha='center', va='top', fontsize=9,
            transform=ax_arr.transAxes, color='#95a5a6', style='italic')

# ── RIGHT: Feature space H ────────────────────────────────────────────────────
ax_out = fig.add_subplot(gs[0, 4], projection='3d')
ax_out.set_facecolor('#ffffff')

ax_out.scatter(Phi1[:, 0], Phi1[:, 1], Phi1[:, 2],
               c='#e07b54', s=12, alpha=0.7, label='Class +1')
ax_out.scatter(Phi2[:, 0], Phi2[:, 1], Phi2[:, 2],
               c='#4a90d9', s=12, alpha=0.7, label='Class -1')

# separating hyperplane
ax_out.plot_surface(xx, yy, zz,
                    alpha=0.25, color='#27ae60', rstride=1, cstride=1,
                    linewidth=0)

ax_out.set_title('Feature Space $\\mathcal{H}$ ($\\mathbb{R}^3$)',
                 fontsize=13, fontweight='bold', pad=10)
ax_out.set_xlabel('$\\varphi_1 = x_1^2$', fontsize=9, labelpad=4)
ax_out.set_ylabel('$\\varphi_2 = \\sqrt{2}x_1x_2$', fontsize=9, labelpad=4)
ax_out.set_zlabel('$\\varphi_3 = x_2^2$', fontsize=9, labelpad=4)
ax_out.legend(fontsize=9, loc='upper left')
ax_out.set_xlim(-0.2, 7)
ax_out.set_ylim(-5, 5)
ax_out.set_zlim(-0.2, 7)
ax_out.view_init(elev=25, azim=-50)

# Green patch for the legend
plane_patch = mpatches.Patch(color='#27ae60', alpha=0.4, label='Separating hyperplane')
ax_out.legend(handles=[ax_out.get_legend_handles_labels()[0][0],
                        ax_out.get_legend_handles_labels()[0][1],
                        plane_patch],
              labels=['Class +1', 'Class -1', 'Separating hyperplane'],
              fontsize=8, loc='upper left')

fig.text(0.72, 0.06, 'Linearly separable', ha='center', fontsize=10,
         color='#27ae60', style='italic', fontweight='bold')

plt.suptitle(
    'Kernel Function: Implicit Mapping from Input Space to Feature Space',
    fontsize=14, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig('kernel_feature_space_diagram.png', dpi=150, bbox_inches='tight',
            facecolor='#f8f9fa')
plt.show()
print('Diagram saved to kernel_feature_space_diagram.png')

### Reading the Diagram

| Panel | Description |
|-------|-------------|
| **Left** (Input space $\mathcal{X}$) | 400 2-D points arranged in two concentric rings. No straight line separates the orange (+1) from the blue (−1) class. The dashed circle marks the ideal decision boundary. |
| **Arrow** | The feature map $\varphi : \mathcal{X} \to \mathcal{H}$.  The kernel trick means we never compute $\varphi(x)$ explicitly; instead we evaluate $K(x, x') = \langle \varphi(x), \varphi(x')\rangle_{\mathcal{H}}$ directly. |
| **Right** (Feature space $\mathcal{H}$) | Each 2-D input is lifted to 3-D via $\varphi(x) = (x_1^2,\,\sqrt{2}x_1 x_2,\,x_2^2)$. The green plane $\varphi_1 + \varphi_3 = r_{\text{mid}}^2$ now **linearly separates** the two classes. |

---

## 5. Verifying the Kernel Identity

We verify numerically that the polynomial kernel $K(x, x') = (x^\top x')^2$ equals the inner product
$\langle \varphi(x), \varphi(x') \rangle_{\mathbb{R}^3}$ for randomly chosen pairs of points.

In [ ]:
# Verify  K(x, x') = <phi(x), phi(x')>  for random sample pairs

def kernel_poly2(a, b):
    """Degree-2 polynomial kernel (no offset): K(a, b) = (a . b)^2"""
    return (a @ b)**2

print(f"{'i':>4}  {'j':>4}  {'K(xi, xj)':>14}  {'<phi(xi), phi(xj)>':>20}  {'match?':>8}")
print('-' * 60)

idx_pairs = [(0, 1), (10, 50), (199, 200), (5, 399), (100, 300)]
for i, j in idx_pairs:
    xi, xj = X[i], X[j]

    k_val   = kernel_poly2(xi, xj)                  # via kernel function
    phi_val = phi(xi[None])[0] @ phi(xj[None])[0]   # via explicit inner product

    match = np.isclose(k_val, phi_val, atol=1e-10)
    print(f"{i:>4}  {j:>4}  {k_val:>14.6f}  {phi_val:>20.6f}  {'✓' if match else '✗':>8}")

print()
print('All pairs satisfy  K(x, x\') = ⟨φ(x), φ(x\')⟩_H ✓')

---

## 6. SVM with Kernel: Classification Demo

We train an SVM with the RBF kernel on the concentric-ring dataset to show that the kernel trick yields a non-linear decision boundary in the original space without ever explicitly computing $\varphi$.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear SVM (no kernel trick)
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_tr, y_tr)
acc_lin = accuracy_score(y_te, svm_lin.predict(X_te))

# SVM with RBF kernel (kernel trick: infinite-dimensional phi)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_tr, y_tr)
acc_rbf = accuracy_score(y_te, svm_rbf.predict(X_te))

# ── decision boundary plot ─────────────────────────────────────────────────────
xx_g, yy_g = np.meshgrid(np.linspace(-3.5, 3.5, 400),
                          np.linspace(-3.5, 3.5, 400))
grid = np.c_[xx_g.ravel(), yy_g.ravel()]

Z_lin = svm_lin.decision_function(grid).reshape(xx_g.shape)
Z_rbf = svm_rbf.decision_function(grid).reshape(xx_g.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#f8f9fa')

titles  = ['Linear SVM (no kernel trick)', 'SVM with RBF Kernel (kernel trick)']
Zs      = [Z_lin, Z_rbf]
accs    = [acc_lin, acc_rbf]

for ax, title, Z, acc in zip(axes, titles, Zs, accs):
    ax.set_facecolor('#ffffff')
    ax.contourf(xx_g, yy_g, Z, levels=[-1, 0, 1],
                colors=['#aec6e8', '#f5c6a0'], alpha=0.45)
    ax.contour(xx_g, yy_g, Z, levels=[0], colors='k', linewidths=2)
    ax.contour(xx_g, yy_g, Z, levels=[-1, 1],
               colors='gray', linewidths=1, linestyles='--')
    ax.scatter(X_te[y_te ==  1, 0], X_te[y_te ==  1, 1],
               c='#e07b54', s=20, edgecolors='k', linewidths=0.4,
               label='Class +1')
    ax.scatter(X_te[y_te == -1, 0], X_te[y_te == -1, 1],
               c='#4a90d9', s=20, edgecolors='k', linewidths=0.4,
               label='Class -1')
    ax.set_title(f'{title}\nTest Accuracy: {acc:.1%}', fontsize=11, fontweight='bold')
    ax.set_xlabel('$x_1$', fontsize=11)
    ax.set_ylabel('$x_2$', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_aspect('equal')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Effect of the Kernel Trick on SVM Decision Boundary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kernel_svm_decision_boundary.png', dpi=150, bbox_inches='tight',
            facecolor='#f8f9fa')
plt.show()
print(f'Linear SVM accuracy : {acc_lin:.1%}')
print(f'RBF kernel SVM accuracy: {acc_rbf:.1%}')

---

## 7. Summary

### The Kernel Function and the Feature Map $\varphi : \mathcal{X} \to \mathcal{H}$

```
Input space X (R^d)          Feature space H (R^D, D >> d or infinite)

  x  ──────────────────── φ(x)  ──┐
                                   ├──►  ⟨φ(x), φ(x')⟩_H  =  K(x, x')
  x' ──────────────────── φ(x') ──┘
                                            ↑
                              computed cheaply in original space
                              (kernel trick: never materialise φ)
```

| Concept | Meaning |
|---------|--------|
| $\mathcal{X}$ | Original (low-dimensional) input space |
| $\mathcal{H}$ | Hilbert feature space (possibly $\infty$-dimensional) |
| $\varphi : \mathcal{X} \to \mathcal{H}$ | Feature map – lifts data to $\mathcal{H}$ |
| $K(x, x')$ | Kernel function – evaluates $\langle \varphi(x), \varphi(x') \rangle_{\mathcal{H}}$ without computing $\varphi$ explicitly |
| Kernel trick | Replace every inner product in an algorithm with $K(\cdot,\cdot)$ to obtain a non-linear method at the cost of a linear one |

**Key identity (repeated for emphasis):**

$$K(x, x') = \langle \varphi(x),\, \varphi(x') \rangle_{\mathcal{H}}$$

**Why this matters for classification:**  
Data that cannot be separated by a hyperplane in $\mathcal{X}$ may become linearly separable in $\mathcal{H}$.  
A support vector machine finds the *optimal* hyperplane in $\mathcal{H}$ using only kernel evaluations, giving a powerful non-linear classifier that operates entirely in the original input space at inference time.

### Relevance to Bird Species Recognition

In the bird species recognition task, feature vectors extracted from image patches (e.g., from a CNN or ViT backbone) may not be linearly separable in the embedding space. Applying an SVM with an RBF kernel to those embeddings implicitly maps them into a higher-dimensional Hilbert space where a linear decision hyperplane can perfectly or near-perfectly separate the 200 species — often outperforming a plain softmax classifier when labelled data are limited.

---

## References

1. Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, 20(3), 273–297.
2. Schölkopf, B., & Smola, A. J. (2002). *Learning with Kernels: Support Vector Machines, Regularization, Optimization, and Beyond*. MIT Press.
3. Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. (Chapter 6: Kernel Methods)
4. Aizerman, M. A., Braverman, E. M., & Rozonoer, L. I. (1964). Theoretical foundations of the potential function method in pattern recognition learning. *Automation and Remote Control*, 25, 821–837.